# SkyLogic MAS — End-to-End Training on Colab GPU

**Pipeline:** YOLOv10 → SegFormer/UNet → SAM → WBF Ensemble → Analysis Report

**Features built in:**
- Auto batch sizing from VRAM
- Resume-from-checkpoint (re-running after a Colab disconnect is safe)
- Continues to next model if any one fails
- Saves everything to `/content/drive/MyDrive/colab_results/`
- Auto-generated analysis & recommendations report at the end

**Before running:**
1. `Runtime → Change runtime type → T4 GPU` (or better)
2. Upload `skylogic_code.zip` to your Drive root (My Drive/skylogic_code.zip)
3. `Runtime → Run all`

---

In [ ]:
# ── CELL 1: Mount Drive + GPU Check ─────────────────────────────────────────
from google.colab import drive
drive.mount('/content/drive')

import torch
CUDA = torch.cuda.is_available()
DEVICE = 'cuda' if CUDA else 'cpu'
print(f'CUDA available : {CUDA}')
if CUDA:
    GPU_NAME = torch.cuda.get_device_name(0)
    VRAM_GB  = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f'GPU            : {GPU_NAME}')
    print(f'VRAM           : {VRAM_GB:.1f} GB')
else:
    print('WARNING: No GPU. Go to Runtime > Change runtime type > GPU')
    GPU_NAME, VRAM_GB = 'CPU', 0.0

In [ ]:
# ── CELL 2: Auto-Tune Batch Sizes Based on VRAM ─────────────────────────────
# Profiles tuned for img=512, FP32. Reduce if OOM during training.
if VRAM_GB >= 35:                # A100 40GB+
    BATCH_YOLO, BATCH_SEG = 64, 32
elif VRAM_GB >= 22:              # A10/A6000
    BATCH_YOLO, BATCH_SEG = 48, 24
elif VRAM_GB >= 15:              # T4 16GB, V100 16GB
    BATCH_YOLO, BATCH_SEG = 16, 8
elif VRAM_GB >= 10:              # P100
    BATCH_YOLO, BATCH_SEG = 12, 6
elif VRAM_GB >= 7:               # K80, lower-end
    BATCH_YOLO, BATCH_SEG = 8, 4
else:                            # CPU or tiny GPU
    BATCH_YOLO, BATCH_SEG = 2, 2

EPOCHS_YOLO = 20
EPOCHS_SEG  = 20

print(f'Auto-tuned for {GPU_NAME} ({VRAM_GB:.1f} GB):')
print(f'  YOLOv10  batch : {BATCH_YOLO}  | epochs: {EPOCHS_YOLO}')
print(f'  SegFormer batch: {BATCH_SEG}   | epochs: {EPOCHS_SEG}')

In [ ]:
# ── CELL 3: Create Output Directories ───────────────────────────────────────
import os
from pathlib import Path

RESULTS_DIR = '/content/drive/MyDrive/colab_results'
for d in ['logs', 'models/yolo', 'models/segformer', 'models/sam',
          'plots', 'predictions', 'checkpoints', 'reports']:
    Path(f'{RESULTS_DIR}/{d}').mkdir(parents=True, exist_ok=True)

print(f'Results dir: {RESULTS_DIR}')
for p in sorted(Path(RESULTS_DIR).iterdir()):
    print(f'  {p.name}/')

In [ ]:
# ── CELL 4: Extract Project Code from Drive ──────────────────────────────────
import zipfile, sys, os
from pathlib import Path

CODE_ZIP    = '/content/drive/MyDrive/skylogic_code.zip'
PROJECT_DIR = '/content/skylogic_project'

if not Path(CODE_ZIP).exists():
    # Try alternate locations
    alts = list(Path('/content/drive/MyDrive').rglob('skylogic_code.zip'))
    if alts:
        CODE_ZIP = str(alts[0])
        print(f'Found zip at alternate location: {CODE_ZIP}')
    else:
        raise FileNotFoundError(
            f'skylogic_code.zip not found on Drive.\n'
            f'Run create_colab_zip.py locally, then upload the zip to your Drive root.'
        )

os.makedirs(PROJECT_DIR, exist_ok=True)
with zipfile.ZipFile(CODE_ZIP, 'r') as zf:
    zf.extractall(PROJECT_DIR)
    print(f'Extracted {len(zf.namelist())} files to {PROJECT_DIR}')

sys.path.insert(0, PROJECT_DIR)
import skylogic
print('skylogic package imported OK')

In [ ]:
# ── CELL 5: Install Dependencies (auto-recover on version conflict) ──────────
import subprocess, sys

DEPS = [
    'ultralytics>=8.3.0', 'transformers>=4.47.0',
    'segmentation-models-pytorch>=0.3.4', 'ensemble-boxes>=1.0.9',
    'rasterio>=1.4.0', 'shapely>=2.0.6', 'geopandas>=1.0.1',
    'fiona>=1.10.1', 'pyproj>=3.7.0', 'qdrant-client>=1.12.0',
    'pydantic-settings>=2.7.0', 'python-dotenv>=1.0.1',
    'pandas>=2.2.0', 'tqdm>=4.67.0', 'opencv-python-headless>=4.10.0',
    'pyyaml', 'matplotlib', 'seaborn', 'timm',
]

def pip_install(pkgs, quiet=True):
    cmd = [sys.executable, '-m', 'pip', 'install'] + (['-q'] if quiet else []) + pkgs
    return subprocess.run(cmd, capture_output=True, text=True)

print('Installing core deps...')
r = pip_install(DEPS)
if r.returncode != 0:
    print('First pass failed, retrying without version pins...')
    relaxed = [p.split('>=')[0].split('==')[0] for p in DEPS]
    r = pip_install(relaxed)
print('Core deps:', 'OK' if r.returncode == 0 else f'WARN -> {r.stderr[-300:]}')

print('Installing segment-anything...')
r = pip_install(['git+https://github.com/facebookresearch/segment-anything.git'])
print('SAM:', 'OK' if r.returncode == 0 else f'WARN -> {r.stderr[-300:]}')

# Sanity import-check
fails = []
for pkg in ['ultralytics', 'transformers', 'segmentation_models_pytorch',
            'ensemble_boxes', 'segment_anything', 'cv2', 'rasterio']:
    try:
        __import__(pkg)
    except ImportError as e:
        fails.append(f'{pkg}: {e}')
if fails:
    print('Import failures:')
    for f in fails:
        print(f'  {f}')
else:
    print('All key imports OK.')

In [ ]:
# ── CELL 6: Environment + Resume Config ─────────────────────────────────────
import os

# Drive dataset paths
DRIVE_DATA_DIR    = '/content/drive/MyDrive/data'
DRIVE_TRAIN_PAT   = f'{DRIVE_DATA_DIR}/patches/train'
DRIVE_VAL_PAT     = f'{DRIVE_DATA_DIR}/patches/val'
PATCHES_META_CSV  = f'{DRIVE_DATA_DIR}/metadata/patches_metadata.csv'
ANNOT_META_CSV    = f'{DRIVE_DATA_DIR}/metadata/annotations_metadata.csv'

# Local SSD
LOCAL_DATA      = '/content/data'
LOCAL_TRAIN_PAT = f'{LOCAL_DATA}/patches/train'
LOCAL_VAL_PAT   = f'{LOCAL_DATA}/patches/val'
LOCAL_YOLO_DIR  = f'{LOCAL_DATA}/yolo'

# Outputs
YOLO_SAVE_DIR = f'{RESULTS_DIR}/models/yolo'
SEG_SAVE_DIR  = f'{RESULTS_DIR}/models/segformer'
SAM_SAVE_DIR  = f'{RESULTS_DIR}/models/sam'
PLOTS_DIR     = f'{RESULTS_DIR}/plots'
LOGS_DIR      = f'{RESULTS_DIR}/logs'
REPORTS_DIR   = f'{RESULTS_DIR}/reports'

for d in [LOCAL_TRAIN_PAT, LOCAL_VAL_PAT, LOCAL_YOLO_DIR,
          YOLO_SAVE_DIR, SEG_SAVE_DIR, SAM_SAVE_DIR,
          PLOTS_DIR, LOGS_DIR, REPORTS_DIR]:
    os.makedirs(d, exist_ok=True)

# Resume flags — set to True to force a rerun even when checkpoint exists
FORCE_RETRAIN_YOLO = False
FORCE_RETRAIN_SEG  = False

os.environ['DATA_DIR']     = DRIVE_DATA_DIR
os.environ['MODELS_DIR']   = f'{RESULTS_DIR}/models'
os.environ['TORCH_DEVICE'] = DEVICE
os.environ['BATCH_SIZE']   = str(BATCH_SEG)

print('Environment configured.')
print(f'  Resume flags  : YOLO={not FORCE_RETRAIN_YOLO}, SEG={not FORCE_RETRAIN_SEG}')

In [ ]:
# ── CELL 7: Verify Dataset on Drive ─────────────────────────────────────────
import pandas as pd
from pathlib import Path

missing = []
for path, label in [(PATCHES_META_CSV, 'patches_metadata.csv'),
                    (ANNOT_META_CSV,   'annotations_metadata.csv'),
                    (DRIVE_TRAIN_PAT,  'patches/train/'),
                    (DRIVE_VAL_PAT,    'patches/val/')]:
    ok = Path(path).exists()
    print(f'  [{"OK" if ok else "MISS"}] {label}')
    if not ok: missing.append(path)

# Auto-fix: search for misplaced data folder
if missing:
    print('\nAttempting auto-discovery of dataset on Drive...')
    candidates = list(Path('/content/drive/MyDrive').glob('**/annotations_metadata.csv'))
    if candidates:
        ANNOT_META_CSV = str(candidates[0])
        DRIVE_DATA_DIR = str(candidates[0].parent.parent)
        PATCHES_META_CSV = f'{DRIVE_DATA_DIR}/metadata/patches_metadata.csv'
        DRIVE_TRAIN_PAT  = f'{DRIVE_DATA_DIR}/patches/train'
        DRIVE_VAL_PAT    = f'{DRIVE_DATA_DIR}/patches/val'
        print(f'Reset DRIVE_DATA_DIR -> {DRIVE_DATA_DIR}')

patches_df = pd.read_csv(PATCHES_META_CSV)
annot_df   = pd.read_csv(ANNOT_META_CSV)
n_train    = len(list(Path(DRIVE_TRAIN_PAT).glob('*.png')))
n_val      = len(list(Path(DRIVE_VAL_PAT).glob('*.png')))
print(f'\nTrain patches : {n_train:,}')
print(f'Val patches   : {n_val:,}')
print(f'Patch meta    : {len(patches_df):,} rows')
print(f'Annot meta    : {len(annot_df):,} rows')
print(f'Classes seen  : {annot_df["class_name"].nunique()}')

In [ ]:
# ── CELL 8: Copy Patches to Local SSD (parallel, resumable) ──────────────────
import shutil
from concurrent.futures import ThreadPoolExecutor, as_completed
from pathlib import Path
from tqdm.auto import tqdm

# Parallel copy: ~10x faster than sequential shutil.copy2 on Drive FUSE.
# Skips files that already exist locally → safe to rerun mid-copy.
def _copy_one(args):
    src, dst = args
    if not dst.exists():
        shutil.copy2(src, dst)
    return dst.stat().st_size

def copy_patches_parallel(src_dir, dst_dir, desc, workers=32):
    files = sorted(Path(src_dir).glob('*.png'))
    Path(dst_dir).mkdir(parents=True, exist_ok=True)
    tasks = [(f, Path(dst_dir) / f.name) for f in files]
    # Quick skip if everything already copied
    pending = [(s, d) for s, d in tasks if not d.exists()]
    if not pending:
        print(f'{desc}: all {len(files):,} files already cached locally')
        return len(files)
    with ThreadPoolExecutor(max_workers=workers) as ex:
        futures = [ex.submit(_copy_one, t) for t in pending]
        for _ in tqdm(as_completed(futures), total=len(futures), desc=desc, unit='img'):
            pass
    return len(files)

n_train_loc = copy_patches_parallel(DRIVE_TRAIN_PAT, LOCAL_TRAIN_PAT, 'Train', workers=32)
n_val_loc   = copy_patches_parallel(DRIVE_VAL_PAT,   LOCAL_VAL_PAT,   'Val',   workers=32)
print(f'\nLocal train: {n_train_loc:,} | val: {n_val_loc:,}')


In [ ]:
# ── CELL 9: Prepare YOLO Labels + data.yaml ──────────────────────────────────
import ast, yaml, shutil, os
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

XVIEW_IDS = sorted([
    11,12,13,15,17,18,19,20,21,23,24,25,26,27,28,29,32,33,34,35,36,37,38,40,
    41,42,44,45,47,49,50,51,52,53,54,55,56,57,59,60,61,62,63,64,65,66,71,72,
    73,74,76,77,79,83,84,86,89,91,93,94
])
XVIEW_NAMES = {
    11:'Fixed-wing Aircraft',12:'Small Aircraft',13:'Cargo Plane',15:'Helicopter',
    17:'Passenger Vehicle',18:'Small Car',19:'Bus',20:'Pickup Truck',21:'Utility Truck',
    23:'Truck',24:'Cargo Truck',25:'Truck w/Box',26:'Truck Tractor',27:'Trailer',
    28:'Truck w/Flatbed',29:'Truck w/Liquid',32:'Crane Truck',33:'Railway Vehicle',
    34:'Passenger Car',35:'Cargo Car',36:'Flat Car',37:'Tank car',38:'Locomotive',
    40:'Maritime Vessel',41:'Motorboat',42:'Sailboat',44:'Tugboat',45:'Barge',
    47:'Fishing Vessel',49:'Ferry',50:'Yacht',51:'Container Ship',52:'Oil Tanker',
    53:'Engineering Vehicle',54:'Tower crane',55:'Container Crane',56:'Reach Stacker',
    57:'Straddle Carrier',59:'Mobile Crane',60:'Dump Truck',61:'Haul Truck',
    62:'Scraper/Tractor',63:'Front loader/Bulldozer',64:'Excavator',65:'Cement Mixer',
    66:'Ground Grader',71:'Hut/Tent',72:'Shed',73:'Building',74:'Aircraft Hangar',
    76:'Damaged Building',77:'Facility',79:'Construction Site',83:'Vehicle Lot',
    84:'Helipad',86:'Storage Tank',89:'Shipping container lot',91:'Shipping Container',
    93:'Pylon',94:'Tower'
}
XVIEW_TO_YOLO    = {xid: i for i, xid in enumerate(XVIEW_IDS)}
YOLO_CLASS_NAMES = [XVIEW_NAMES[xid] for xid in XVIEW_IDS]
IMG_SIZE = 512

YOLO_IMG_TRAIN = f'{LOCAL_YOLO_DIR}/images/train'
YOLO_IMG_VAL   = f'{LOCAL_YOLO_DIR}/images/val'
YOLO_LBL_TRAIN = f'{LOCAL_YOLO_DIR}/labels/train'
YOLO_LBL_VAL   = f'{LOCAL_YOLO_DIR}/labels/val'
for d in [YOLO_IMG_TRAIN, YOLO_IMG_VAL, YOLO_LBL_TRAIN, YOLO_LBL_VAL]:
    os.makedirs(d, exist_ok=True)

# Symlink images
for src, dst in [(LOCAL_TRAIN_PAT, YOLO_IMG_TRAIN), (LOCAL_VAL_PAT, YOLO_IMG_VAL)]:
    for f in Path(src).glob('*.png'):
        d = Path(dst) / f.name
        if not d.exists():
            try: d.symlink_to(f.resolve())
            except Exception: shutil.copy2(f, d)

train_set = {p.name for p in Path(LOCAL_TRAIN_PAT).glob('*.png')}
val_set   = {p.name for p in Path(LOCAL_VAL_PAT).glob('*.png')}

written, skipped = 0, 0
for patch_name, group in tqdm(annot_df.groupby('patch_filename'), desc='Labels'):
    if patch_name in train_set:   lbl_path = Path(YOLO_LBL_TRAIN) / patch_name.replace('.png', '.txt')
    elif patch_name in val_set:   lbl_path = Path(YOLO_LBL_VAL) / patch_name.replace('.png', '.txt')
    else: skipped += 1; continue
    lines = []
    for _, row in group.iterrows():
        cid = int(row['class_id'])
        if cid not in XVIEW_TO_YOLO: continue
        bbox = ast.literal_eval(row['bbox']) if isinstance(row['bbox'], str) else row['bbox']
        x1,y1,x2,y2 = bbox
        cx = max(0.0, min(1.0, ((x1+x2)/2)/IMG_SIZE))
        cy = max(0.0, min(1.0, ((y1+y2)/2)/IMG_SIZE))
        w  = max(0.001, min(1.0, (x2-x1)/IMG_SIZE))
        h  = max(0.001, min(1.0, (y2-y1)/IMG_SIZE))
        lines.append(f'{XVIEW_TO_YOLO[cid]} {cx:.6f} {cy:.6f} {w:.6f} {h:.6f}')
    if lines:
        lbl_path.write_text('\n'.join(lines)); written += 1

DATA_YAML_PATH = f'{LOCAL_YOLO_DIR}/data.yaml'
with open(DATA_YAML_PATH, 'w') as f:
    yaml.dump({'path': LOCAL_YOLO_DIR, 'train': 'images/train', 'val': 'images/val',
               'nc': len(XVIEW_IDS), 'names': YOLO_CLASS_NAMES}, f)
print(f'Labels: {written} written, {skipped} skipped | data.yaml: {DATA_YAML_PATH}')

In [ ]:
# ── CELL 10: Build Segmentation Dataset ──────────────────────────────────────
import ast, numpy as np, torch
from pathlib import Path
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

XVIEW_TO_SEG = {
    71:1,72:1,73:1,74:1,77:1, 76:2,
    17:3,18:3,19:3,20:3,21:3,23:3,24:3,25:3,26:3,27:3,28:3,29:3,32:3,83:3,
    40:3,41:3,42:3,44:3,45:3,47:3,49:3,50:3,51:3,52:3,
    53:8,54:8,55:8,56:8,57:8,59:8,60:8,61:8,62:8,63:8,64:8,65:8,66:8,79:8,
    86:9,89:9,91:9,
}
SEG_CLASS_NAMES = {0:'background',1:'building',2:'damaged_building',3:'vehicle',
                   4:'road',5:'vegetation',6:'water_flood',7:'debris_rubble',
                   8:'construction',9:'container'}

IMG_TF = transforms.Compose([
    transforms.Resize((512,512)), transforms.ToTensor(),
    transforms.Normalize(mean=[0.485,0.456,0.406], std=[0.229,0.224,0.225]),
])

class SatSegDataset(Dataset):
    def __init__(self, patches_dir, annot_df):
        self.dir = Path(patches_dir)
        avail = {p.name for p in self.dir.glob('*.png')}
        self.by_patch = {n: g for n, g in annot_df.groupby('patch_filename') if n in avail}
        self.patches = sorted(avail)
    def __len__(self): return len(self.patches)
    def __getitem__(self, i):
        name = self.patches[i]
        img = Image.open(self.dir/name).convert('RGB')
        w, h = img.size
        img_t = IMG_TF(img)
        mask = np.zeros((h, w), dtype=np.uint8)
        if name in self.by_patch:
            for _, row in self.by_patch[name].iterrows():
                sc = XVIEW_TO_SEG.get(int(row['class_id']), 0)
                if sc == 0: continue
                bb = ast.literal_eval(row['bbox']) if isinstance(row['bbox'], str) else row['bbox']
                x1,y1,x2,y2 = (max(0,int(v)) for v in bb)
                x2, y2 = min(w, x2), min(h, y2)
                mask[y1:y2, x1:x2] = sc
        m_r = Image.fromarray(mask).resize((512,512), Image.NEAREST)
        return {'image': img_t, 'mask': torch.from_numpy(np.array(m_r)).long(), 'name': name}

train_seg_ds = SatSegDataset(LOCAL_TRAIN_PAT, annot_df)
val_seg_ds   = SatSegDataset(LOCAL_VAL_PAT,   annot_df)
train_seg_dl = DataLoader(train_seg_ds, batch_size=BATCH_SEG, shuffle=True,  num_workers=2, pin_memory=True)
val_seg_dl   = DataLoader(val_seg_ds,   batch_size=BATCH_SEG, shuffle=False, num_workers=2, pin_memory=True)
print(f'Seg dataset: train={len(train_seg_ds):,} | val={len(val_seg_ds):,}')

## Model 1 — YOLOv10

In [ ]:
# ── CELL 11: YOLOv10 — Train (resume + checkpoint-aware) ─────────────────────
import time, json
from pathlib import Path

MODEL_RESULTS = {}
yolo_status, yolo_metrics, yolo_error = 'not_started', {}, None
yolo_best_weights = None

# Look for best.pt (training already complete) or last.pt (interrupted training)
best_pts = list(Path(YOLO_SAVE_DIR).rglob('best.pt'))
last_pts = list(Path(YOLO_SAVE_DIR).rglob('last.pt'))

if best_pts and not FORCE_RETRAIN_YOLO:
    yolo_best_weights = str(best_pts[-1])
    yolo_status = 'resumed'
    print(f'Existing YOLO best.pt found, skipping training: {yolo_best_weights}')
elif last_pts and not FORCE_RETRAIN_YOLO:
    # Interrupted training → resume from last.pt with Ultralytics native resume
    resume_ckpt = str(last_pts[-1])
    print(f'Found interrupted YOLO training at: {resume_ckpt}')
    print('Resuming from last.pt (Ultralytics native resume=True)...')
    try:
        from ultralytics import YOLO
        m = YOLO(resume_ckpt)
        t0 = time.time()
        m.train(resume=True)
        train_time_min = (time.time() - t0) / 60
        yolo_metrics['train_time_min'] = round(train_time_min, 2)
        yolo_metrics['resumed_from'] = resume_ckpt
        best_pts = list(Path(YOLO_SAVE_DIR).rglob('best.pt'))
        yolo_best_weights = str(best_pts[-1]) if best_pts else resume_ckpt
        yolo_status = 'completed'
        print(f'YOLO resume completed in {train_time_min:.1f} min -> {yolo_best_weights}')
    except Exception as e:
        import traceback
        yolo_error = traceback.format_exc()
        yolo_status = 'failed'
        with open(f'{LOGS_DIR}/yolo_error.txt', 'w') as f: f.write(yolo_error)
        print(f'YOLO RESUME FAILED (continuing):\n{yolo_error[-500:]}')
else:
    try:
        from skylogic.agents.detector import DetectorAgent
        detector = DetectorAgent(model_path=None, device=DEVICE, num_classes=60)
        t0 = time.time()
        detector.train(data_yaml=DATA_YAML_PATH,
                       epochs=EPOCHS_YOLO, batch_size=BATCH_YOLO,
                       img_size=512, save_dir=YOLO_SAVE_DIR)
        train_time_min = (time.time() - t0) / 60
        yolo_metrics['train_time_min'] = round(train_time_min, 2)
        best_pts = list(Path(YOLO_SAVE_DIR).rglob('best.pt'))
        yolo_best_weights = str(best_pts[-1]) if best_pts else None
        yolo_status = 'completed'
        print(f'YOLO trained in {train_time_min:.1f} min -> {yolo_best_weights}')
    except Exception as e:
        import traceback
        yolo_error = traceback.format_exc()
        yolo_status = 'failed'
        with open(f'{LOGS_DIR}/yolo_error.txt', 'w') as f: f.write(yolo_error)
        print(f'YOLO FAILED (continuing):\n{yolo_error[-500:]}')


In [ ]:
# ── CELL 12: YOLOv10 — Evaluate ──────────────────────────────────────────────
import time, json
from pathlib import Path

if yolo_best_weights and yolo_status in ('completed', 'resumed'):
    try:
        from ultralytics import YOLO
        m = YOLO(yolo_best_weights)
        t0 = time.time()
        v = m.val(data=DATA_YAML_PATH, imgsz=512, batch=BATCH_YOLO, device=DEVICE, verbose=False)
        infer_ms = (time.time()-t0)*1000 / max(1, len(list(Path(YOLO_IMG_VAL).glob('*.png'))))
        yolo_metrics.update({
            'mAP50':     round(float(v.box.map50), 4),
            'mAP50_95':  round(float(v.box.map),   4),
            'precision': round(float(v.box.mp),    4),
            'recall':    round(float(v.box.mr),    4),
            'inference_ms': round(infer_ms, 2),
        })
        p, r = yolo_metrics['precision'], yolo_metrics['recall']
        yolo_metrics['f1'] = round(2*p*r / max(1e-6, p+r), 4)
        with open(f'{LOGS_DIR}/yolo_metrics.json', 'w') as f: json.dump(yolo_metrics, f, indent=2)
        for k, val in yolo_metrics.items(): print(f'  {k:<16}: {val}')
    except Exception as e:
        import traceback
        err = traceback.format_exc()
        print(f'YOLO eval failed: {err[-400:]}')
        with open(f'{LOGS_DIR}/yolo_eval_error.txt', 'w') as f: f.write(err)
else:
    print(f'Skipping YOLO eval (status={yolo_status})')

MODEL_RESULTS['YOLOv10'] = {'status': yolo_status, 'metrics': yolo_metrics, 'error': yolo_error}

In [ ]:
# ── CELL 13: YOLOv10 — Save Prediction Plots ─────────────────────────────────
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path

if yolo_best_weights:
    try:
        from ultralytics import YOLO
        m = YOLO(yolo_best_weights)
        imgs = sorted(Path(YOLO_IMG_VAL).glob('*.png'))[:4]
        if imgs:
            fig, axes = plt.subplots(1, len(imgs), figsize=(5*len(imgs), 5))
            if len(imgs) == 1: axes = [axes]
            for ax, ip in zip(axes, imgs):
                r = m.predict(str(ip), conf=0.25, verbose=False)[0]
                ax.imshow(Image.open(ip).convert('RGB'))
                if r.boxes:
                    for b in r.boxes:
                        x1,y1,x2,y2 = b.xyxy[0].cpu().numpy()
                        c = int(b.cls[0]); conf = float(b.conf[0])
                        ax.add_patch(mpatches.Rectangle((x1,y1), x2-x1, y2-y1,
                                     linewidth=1.5, edgecolor='lime', facecolor='none'))
                        ax.text(x1, y1-2, f'{YOLO_CLASS_NAMES[c][:8]} {conf:.2f}',
                                fontsize=6, color='lime', backgroundcolor='black')
                ax.set_title(ip.name[:20], fontsize=8); ax.axis('off')
            plt.suptitle('YOLOv10 Predictions (val)', fontsize=12)
            plt.tight_layout()
            plt.savefig(f'{PLOTS_DIR}/yolo_predictions.png', dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        print(f'Plot generation failed: {e}')

## Model 2 — SegFormer / U-Net

In [ ]:
# ── CELL 14: SegFormer — Train (resume-aware) ────────────────────────────────
import time, json
from pathlib import Path

seg_status, seg_metrics, seg_error = 'not_started', {}, None
best_seg_ckpt = f'{SEG_SAVE_DIR}/best.pth'

if Path(best_seg_ckpt).exists() and not FORCE_RETRAIN_SEG:
    seg_status = 'resumed'
    print(f'Existing SegFormer checkpoint found: {best_seg_ckpt}')
else:
    try:
        from skylogic.agents.segmentor import SegmentorAgent
        segmentor = SegmentorAgent(model_path=None, device=DEVICE, num_classes=10, img_size=512)
        t0 = time.time()
        out = segmentor.train(train_loader=train_seg_dl, val_loader=val_seg_dl,
                              epochs=EPOCHS_SEG, lr=1e-4, save_dir=SEG_SAVE_DIR)
        ttm = (time.time() - t0) / 60
        seg_metrics = {
            'train_time_min':  round(ttm, 2),
            'best_loss':       round(out.get('best_loss', -1), 4),
            'final_train_loss': round(out['history']['train_loss'][-1] if out['history']['train_loss'] else -1, 4),
            'final_val_loss':   round(out['history']['val_loss'][-1] if out['history']['val_loss'] else -1, 4),
            'model_kind':       segmentor._model_kind,
        }
        seg_status = 'completed'
        print(f'SegFormer trained in {ttm:.1f} min')
    except Exception as e:
        import traceback
        seg_error = traceback.format_exc()
        seg_status = 'failed'
        with open(f'{LOGS_DIR}/segmentor_error.txt', 'w') as f: f.write(seg_error)
        print(f'SegFormer FAILED (continuing):\n{seg_error[-500:]}')

In [ ]:
# ── CELL 15: SegFormer — Evaluate (mIoU) + Plots ─────────────────────────────
import time, json, numpy as np
import torch, torch.nn.functional as F
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

if Path(best_seg_ckpt).exists():
    try:
        from skylogic.agents.segmentor import SegmentorAgent
        seg_eval = SegmentorAgent(model_path=best_seg_ckpt, device=DEVICE, num_classes=10)
        n_cls, cm = 10, np.zeros((10, 10), dtype=np.int64)
        t0 = time.time()
        with torch.no_grad():
            for batch in val_seg_dl:
                imgs = batch['image'].to(DEVICE)
                masks = batch['mask'].numpy()
                if seg_eval._model_kind == 'segformer':
                    logits = F.interpolate(seg_eval.model(pixel_values=imgs).logits,
                                            size=(512, 512), mode='bilinear', align_corners=False)
                else:
                    logits = seg_eval.model(imgs)
                preds = logits.argmax(dim=1).cpu().numpy()
                for p, g in zip(preds, masks):
                    for c in range(n_cls):
                        for pp in range(n_cls):
                            cm[c, pp] += int(np.sum((g == c) & (p == pp)))
        infer_ms = (time.time()-t0)*1000 / max(1, len(val_seg_ds))

        ious = []
        for c in range(n_cls):
            tp = cm[c, c]; fn = cm[c].sum()-tp; fp = cm[:, c].sum()-tp
            ious.append(round(float(tp / max(1, tp+fp+fn)), 4))
        miou = round(float(np.mean(ious)), 4)
        seg_metrics.update({
            'mIoU': miou, 'inference_ms': round(infer_ms, 2),
            'iou_per_class': {SEG_CLASS_NAMES[i]: ious[i] for i in range(n_cls)},
        })
        with open(f'{LOGS_DIR}/segmentor_metrics.json', 'w') as f: json.dump(seg_metrics, f, indent=2)
        print(f'mIoU: {miou}')
        for name, v in seg_metrics['iou_per_class'].items():
            print(f'  {name:<20}: {v}')

        PALETTE = [[0,0,0],[0,128,255],[255,0,0],[0,255,0],[128,128,0],
                   [0,200,0],[64,200,255],[150,75,0],[255,165,0],[128,0,128]]
        imgs_v = sorted(Path(LOCAL_VAL_PAT).glob('*.png'))[:3]
        if imgs_v:
            fig, axes = plt.subplots(len(imgs_v), 2, figsize=(10, 4*len(imgs_v)))
            if len(imgs_v) == 1: axes = [axes]
            for row, ip in zip(axes, imgs_v):
                res = seg_eval.predict(str(ip))
                img = Image.open(ip).convert('RGB').resize((512, 512))
                m = res['mask']
                rgb = np.array([PALETTE[c] for c in m.flatten()], dtype=np.uint8).reshape(512, 512, 3)
                row[0].imshow(img); row[0].set_title(ip.name[:20], fontsize=8); row[0].axis('off')
                row[1].imshow(rgb); row[1].set_title('mask', fontsize=8); row[1].axis('off')
            plt.suptitle('SegFormer Predictions', fontsize=12); plt.tight_layout()
            plt.savefig(f'{PLOTS_DIR}/segformer_predictions.png', dpi=150, bbox_inches='tight')
            plt.show()
    except Exception as e:
        import traceback
        err = traceback.format_exc()
        print(f'Seg eval failed: {err[-400:]}')
        with open(f'{LOGS_DIR}/segmentor_eval_error.txt', 'w') as f: f.write(err)

MODEL_RESULTS['SegFormer'] = {'status': seg_status, 'metrics': seg_metrics, 'error': seg_error}

## Model 3 — SAM

In [ ]:
# ── CELL 16: SAM — Download + Inference (Drive-cached) ───────────────────────
import os, time, json, shutil, numpy as np
import matplotlib.pyplot as plt
from PIL import Image
from pathlib import Path

sam_status, sam_metrics, sam_error = 'not_started', {}, None

# Use ViT-B if VRAM tight, ViT-H otherwise
SAM_MODEL_TYPE = 'vit_h' if VRAM_GB >= 14 else 'vit_b'
SAM_URL = {
    'vit_b': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_b_01ec64.pth',
    'vit_h': 'https://dl.fbaipublicfiles.com/segment_anything/sam_vit_h_4b8939.pth',
}[SAM_MODEL_TYPE]

# Drive-side cache: saves ~5 min per cold start on rerun
SAM_DRIVE_CKPT = f'{SAM_SAVE_DIR}/sam_{SAM_MODEL_TYPE}.pth'
SAM_LOCAL_CKPT = f'/content/models/sam_{SAM_MODEL_TYPE}.pth'
os.makedirs('/content/models', exist_ok=True)

if Path(SAM_LOCAL_CKPT).exists():
    print(f'SAM checkpoint already on /content: {SAM_LOCAL_CKPT}')
elif Path(SAM_DRIVE_CKPT).exists():
    print(f'Copying SAM checkpoint from Drive cache: {SAM_DRIVE_CKPT}')
    shutil.copy2(SAM_DRIVE_CKPT, SAM_LOCAL_CKPT)
else:
    print(f'Downloading SAM {SAM_MODEL_TYPE} from facebook (no Drive cache)...')
    !wget -q --show-progress '{SAM_URL}' -O '{SAM_LOCAL_CKPT}'
    # Cache to Drive for next session
    print(f'Caching SAM checkpoint to Drive: {SAM_DRIVE_CKPT}')
    try:
        shutil.copy2(SAM_LOCAL_CKPT, SAM_DRIVE_CKPT)
    except Exception as e:
        print(f'(Drive cache failed, will redownload next time: {e})')

SAM_CKPT = SAM_LOCAL_CKPT
print(f'SAM checkpoint: {SAM_CKPT} ({Path(SAM_CKPT).stat().st_size/1e6:.1f} MB)')

try:
    from skylogic.agents.sam_agent import SAMAgent
    sam_agent = SAMAgent(checkpoint_path=SAM_CKPT, model_type=SAM_MODEL_TYPE, device=DEVICE)
    if sam_agent.predictor is None: raise RuntimeError('SAM predictor failed to load')

    imgs = sorted(Path(LOCAL_VAL_PAT).glob('*.png'))[:6] or sorted(Path(LOCAL_TRAIN_PAT).glob('*.png'))[:6]
    fig, axes = plt.subplots(2, 3, figsize=(15, 10))
    axes = axes.flatten()
    total_ms, ok_count = 0.0, 0
    for i, ip in enumerate(imgs[:6]):
        t0 = time.time()
        if not sam_agent.set_image(str(ip)):
            axes[i].set_title(f'{ip.name[:15]}\n(skip)', fontsize=7); axes[i].axis('off'); continue
        r = sam_agent.predict_click([[256, 256]], [1], multimask_output=True)
        ms = (time.time()-t0)*1000; total_ms += ms; ok_count += 1
        img = np.array(Image.open(ip).convert('RGB').resize((512, 512)))
        axes[i].imshow(img)
        if r.get('best_mask') is not None:
            m = r['best_mask']; m = np.array(m) if isinstance(m, list) else m
            ov = np.zeros((*m.shape, 4), dtype=np.uint8); ov[m > 0] = [0, 255, 0, 120]
            axes[i].imshow(ov); axes[i].plot(256, 256, 'r*', markersize=10)
            axes[i].set_title(f'{ip.name[:15]}\nscore={r["best_score"]:.3f} | {ms:.0f}ms', fontsize=7)
        else:
            axes[i].set_title(f'{ip.name[:15]}\nno mask', fontsize=7)
        axes[i].axis('off')
    plt.suptitle(f'SAM {SAM_MODEL_TYPE.upper()} click predictions', fontsize=12); plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/sam_predictions.png', dpi=150, bbox_inches='tight'); plt.show()

    sam_metrics = {'model_type': SAM_MODEL_TYPE, 'samples_run': ok_count,
                    'inference_ms_avg': round(total_ms / max(1, ok_count), 2),
                    'drive_cached': Path(SAM_DRIVE_CKPT).exists(),
                    'note': 'Inference only (no training)'}
    with open(f'{LOGS_DIR}/sam_metrics.json', 'w') as f: json.dump(sam_metrics, f, indent=2)
    sam_status = 'completed'
    print(f'SAM avg inference: {sam_metrics["inference_ms_avg"]:.1f} ms/image')
except Exception as e:
    import traceback
    sam_error = traceback.format_exc()
    sam_status = 'failed'
    with open(f'{LOGS_DIR}/sam_error.txt', 'w') as f: f.write(sam_error)
    print(f'SAM FAILED (continuing):\n{sam_error[-500:]}')

MODEL_RESULTS['SAM'] = {'status': sam_status, 'metrics': sam_metrics, 'error': sam_error}


## Model 4 — WBF Ensemble

In [ ]:
# ── CELL 17: WBF Ensemble (YOLO + SegFormer) ─────────────────────────────────
import time, json, numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from PIL import Image
from pathlib import Path

ens_status, ens_metrics, ens_error = 'not_started', {}, None

try:
    from skylogic.ensemble.wbf_fusion import WBFFusion
    from skylogic.agents.detector  import DetectorAgent
    from skylogic.agents.segmentor import SegmentorAgent
    det  = DetectorAgent(model_path=yolo_best_weights if Path(yolo_best_weights).exists() else None,
                          device=DEVICE)
    seg  = SegmentorAgent(model_path=best_seg_ckpt if Path(best_seg_ckpt).exists() else None,
                           device=DEVICE, num_classes=10)
    fuse = WBFFusion(iou_threshold=0.5, weights=[2.0, 1.0])

    imgs = sorted(Path(LOCAL_VAL_PAT).glob('*.png'))[:8] or sorted(Path(LOCAL_TRAIN_PAT).glob('*.png'))[:8]
    dcs, scs, fcs, total_ms = [], [], [], 0.0
    preds_log = []
    for ip in imgs:
        t0 = time.time()
        d  = det.predict(str(ip))
        sr = seg.predict(str(ip))
        sb = seg.mask_to_bboxes(sr['mask']) if sr.get('mask') is not None else []
        f  = fuse.fuse(d, sb, image_width=512, image_height=512)
        ms = (time.time() - t0) * 1000; total_ms += ms
        dcs.append(len(d)); scs.append(len(sb)); fcs.append(len(f))
        preds_log.append({'image': ip.name, 'det': len(d), 'seg': len(sb),
                          'fused': len(f), 'ms': round(ms, 2)})

    ens_metrics = {
        'samples_run':       len(imgs),
        'avg_detections':    round(np.mean(dcs), 2),
        'avg_seg_bboxes':    round(np.mean(scs), 2),
        'avg_fused_boxes':   round(np.mean(fcs), 2),
        'inference_ms_avg':  round(total_ms / max(1, len(imgs)), 2),
    }
    with open(f'{LOGS_DIR}/ensemble_metrics.json', 'w') as f: json.dump(ens_metrics, f, indent=2)
    with open(f'{LOGS_DIR}/ensemble_preds.json', 'w') as f: json.dump(preds_log, f, indent=2)

    # Visualization on first image
    ip = imgs[0]
    d  = det.predict(str(ip)); sr = seg.predict(str(ip))
    sb = seg.mask_to_bboxes(sr['mask']) if sr.get('mask') is not None else []
    f  = fuse.fuse(d, sb)
    img = np.array(Image.open(ip).convert('RGB').resize((512, 512)))
    fig, axes = plt.subplots(1, 3, figsize=(18, 6))
    for ax, preds, title, c in [(axes[0], d, f'YOLO ({len(d)})', 'lime'),
                                 (axes[1], sb, f'SegFormer bbox ({len(sb)})', 'cyan'),
                                 (axes[2], f, f'WBF Fused ({len(f)})', 'yellow')]:
        ax.imshow(img)
        for p in preds:
            x1,y1,x2,y2 = p['bbox']
            ax.add_patch(mpatches.Rectangle((x1,y1), x2-x1, y2-y1,
                                            linewidth=1.5, edgecolor=c, facecolor='none'))
        ax.set_title(title, fontsize=9); ax.axis('off')
    plt.suptitle(f'WBF Ensemble — {ip.name}', fontsize=12); plt.tight_layout()
    plt.savefig(f'{PLOTS_DIR}/ensemble_fusion.png', dpi=150, bbox_inches='tight'); plt.show()

    ens_status = 'completed'
    for k, v in ens_metrics.items(): print(f'  {k:<22}: {v}')
except Exception as e:
    import traceback
    ens_error = traceback.format_exc()
    ens_status = 'failed'
    with open(f'{LOGS_DIR}/ensemble_error.txt', 'w') as f: f.write(ens_error)
    print(f'Ensemble FAILED:\n{ens_error[-500:]}')

MODEL_RESULTS['WBF_Ensemble'] = {'status': ens_status, 'metrics': ens_metrics, 'error': ens_error}

## Summary, Analysis & Report

In [ ]:
# ── CELL 18: Build Comparison Table + CSV ────────────────────────────────────
import pandas as pd

def _g(d, *ks, default='N/A'):
    for k in ks:
        if not isinstance(d, dict): return default
        d = d.get(k, default)
    return d

rows = [
    {'Model': 'YOLOv10 (Agent A)', 'Task': 'Object Detection',
     'Status':    _g(MODEL_RESULTS, 'YOLOv10', 'status'),
     'mAP@0.5':   _g(MODEL_RESULTS, 'YOLOv10', 'metrics', 'mAP50'),
     'mAP@0.5:0.95': _g(MODEL_RESULTS, 'YOLOv10', 'metrics', 'mAP50_95'),
     'Precision': _g(MODEL_RESULTS, 'YOLOv10', 'metrics', 'precision'),
     'Recall':    _g(MODEL_RESULTS, 'YOLOv10', 'metrics', 'recall'),
     'F1':        _g(MODEL_RESULTS, 'YOLOv10', 'metrics', 'f1'),
     'mIoU':      'N/A',
     'Train_min': _g(MODEL_RESULTS, 'YOLOv10', 'metrics', 'train_time_min'),
     'Inf_ms':    _g(MODEL_RESULTS, 'YOLOv10', 'metrics', 'inference_ms'),
     'Notes':     f'60 cls | batch={BATCH_YOLO}'},
    {'Model': 'SegFormer (Agent B)', 'Task': 'Semantic Segmentation',
     'Status':    _g(MODEL_RESULTS, 'SegFormer', 'status'),
     'mAP@0.5':   'N/A', 'mAP@0.5:0.95': 'N/A',
     'Precision': 'N/A', 'Recall': 'N/A', 'F1': 'N/A',
     'mIoU':      _g(MODEL_RESULTS, 'SegFormer', 'metrics', 'mIoU'),
     'Train_min': _g(MODEL_RESULTS, 'SegFormer', 'metrics', 'train_time_min'),
     'Inf_ms':    _g(MODEL_RESULTS, 'SegFormer', 'metrics', 'inference_ms'),
     'Notes':     f'10 cls | {_g(MODEL_RESULTS, "SegFormer", "metrics", "model_kind")}'},
    {'Model': f'SAM {SAM_MODEL_TYPE.upper()} (Agent C)', 'Task': 'Click Segmentation',
     'Status':    _g(MODEL_RESULTS, 'SAM', 'status'),
     'mAP@0.5':   'N/A', 'mAP@0.5:0.95': 'N/A',
     'Precision': 'N/A', 'Recall': 'N/A', 'F1': 'N/A', 'mIoU': 'N/A',
     'Train_min': 'N/A',
     'Inf_ms':    _g(MODEL_RESULTS, 'SAM', 'metrics', 'inference_ms_avg'),
     'Notes':     f'Zero-shot | {_g(MODEL_RESULTS, "SAM", "metrics", "samples_run")} samples'},
    {'Model': 'WBF Ensemble', 'Task': 'Fusion',
     'Status':    _g(MODEL_RESULTS, 'WBF_Ensemble', 'status'),
     'mAP@0.5':   'N/A', 'mAP@0.5:0.95': 'N/A',
     'Precision': 'N/A', 'Recall': 'N/A', 'F1': 'N/A', 'mIoU': 'N/A',
     'Train_min': 'N/A',
     'Inf_ms':    _g(MODEL_RESULTS, 'WBF_Ensemble', 'metrics', 'inference_ms_avg'),
     'Notes':     f'IoU=0.5 | weights=[2,1]'},
]
df = pd.DataFrame(rows)
df.to_csv(f'{RESULTS_DIR}/model_comparison_summary.csv', index=False)
print(df.to_string(index=False))
print(f'\nSaved: {RESULTS_DIR}/model_comparison_summary.csv')

In [ ]:
# ── CELL 19: Auto Analysis & Recommendations Report ──────────────────────────
from datetime import datetime

def _g(d, *ks, default=None):
    for k in ks:
        if not isinstance(d, dict): return default
        d = d.get(k, default)
    return d

def _is_num(x):
    return isinstance(x, (int, float)) and not isinstance(x, bool)

# ── Identify weaknesses ──
weaknesses = []
recommendations = []
best_model = None
best_score = -1

y = _g(MODEL_RESULTS, 'YOLOv10', 'metrics') or {}
s = _g(MODEL_RESULTS, 'SegFormer', 'metrics') or {}
sam = _g(MODEL_RESULTS, 'SAM', 'metrics') or {}
ens = _g(MODEL_RESULTS, 'WBF_Ensemble', 'metrics') or {}

# YOLO analysis
if _is_num(y.get('mAP50')):
    mp = y['mAP50']; mr = y.get('recall', 0); mpr = y.get('precision', 0)
    if mp > best_score: best_score, best_model = mp, 'YOLOv10'
    if mp < 0.30:
        weaknesses.append(f'YOLOv10 mAP@0.5 is low ({mp:.3f}) — model is underfitting or data is hard.')
        recommendations += [
            'Train YOLOv10 for more epochs (50-100) — 20 is likely under-trained for 60 classes.',
            'Use a larger YOLO model (yolov8s.pt or yolov8m.pt) instead of yolov8n.',
            'Add augmentations: mosaic=1.0, mixup=0.15, copy_paste=0.3 (already on by default in ultralytics).',
            'Address class imbalance: oversample rare classes (Helipad, Ferry, Cement Mixer) or use class-weighted loss.',
        ]
    elif mp < 0.55:
        weaknesses.append(f'YOLOv10 mAP@0.5 is moderate ({mp:.3f}) — room for clear improvement.')
        recommendations += [
            'Bump epochs from 20 to 50, lr0=0.01, cos_lr=True.',
            'Switch to yolov8s.pt for capacity, keep batch size the same.',
        ]
    if mr < 0.50 and _is_num(mr):
        weaknesses.append(f'YOLOv10 recall is low ({mr:.3f}) — many objects missed.')
        recommendations.append('Lower confidence threshold during inference (try conf=0.15) and re-evaluate.')
    if _is_num(mpr) and mpr < 0.50:
        weaknesses.append(f'YOLOv10 precision is low ({mpr:.3f}) — many false positives.')
        recommendations.append('Raise NMS IoU threshold to 0.6 and use TTA at inference (`augment=True`).')

# SegFormer analysis
if _is_num(s.get('mIoU')):
    miou = s['mIoU']
    if miou > best_score: best_score, best_model = miou, 'SegFormer'
    if miou < 0.20:
        weaknesses.append(f'SegFormer mIoU very low ({miou:.3f}) — masks-from-bboxes is a weak proxy ground truth.')
        recommendations += [
            'Use real polygon segmentation labels instead of bbox-filled masks (use xView GeoJSON polygons).',
            'OR: use SAM to auto-generate pseudo-masks from the YOLO boxes and train SegFormer on those.',
            'Try the proper transformers SegFormer with imagenet pretrained weights instead of random init.',
        ]
    elif miou < 0.40:
        weaknesses.append(f'SegFormer mIoU moderate ({miou:.3f}).')
        recommendations += [
            'Increase epochs to 50, use lr=5e-5 with cosine schedule and warmup.',
            'Use a class-balanced loss (e.g. focal or weighted CrossEntropy).',
        ]
    iou_per = s.get('iou_per_class', {})
    if iou_per:
        worst = sorted(iou_per.items(), key=lambda x: x[1])[:3]
        if worst:
            weaknesses.append(f'Worst seg classes: {worst}')
            recommendations.append('Drop or merge segmentation classes with zero IoU — they bring the mean down.')

# Status checks
for name, key in [('YOLOv10', 'YOLOv10'), ('SegFormer', 'SegFormer'),
                   ('SAM', 'SAM'), ('WBF Ensemble', 'WBF_Ensemble')]:
    if _g(MODEL_RESULTS, key, 'status') == 'failed':
        weaknesses.append(f'{name} run FAILED — see logs/{key.lower()}_error.txt')
        recommendations.append(f'Investigate {name} error log and re-run that stage.')

# General recommendations if nothing else triggered
if not weaknesses:
    weaknesses.append('No major weaknesses detected from this single run.')
if not recommendations:
    recommendations += [
        'Add test-time augmentation (TTA) to YOLO inference (augment=True).',
        'Run k-fold cross-validation to confirm metrics are stable.',
    ]

if best_model is None:
    best_model = 'YOLOv10 (default — main task is detection)'

# ── Build Markdown report ──
ts = datetime.now().strftime('%Y-%m-%d %H:%M')
lines = [
    f'# SkyLogic MAS — Training Report',
    f'_Generated: {ts}_  |  _Device: {GPU_NAME} ({VRAM_GB:.1f} GB)_',
    '',
    '## 1. Model Performance Summary',
    '',
    df.to_markdown(index=False) if hasattr(df, 'to_markdown') else df.to_string(index=False),
    '',
    '## 2. Training Times',
    '',
    f'- YOLOv10  : **{y.get("train_time_min", "N/A")} min** ({EPOCHS_YOLO} epochs, batch {BATCH_YOLO})',
    f'- SegFormer: **{s.get("train_time_min", "N/A")} min** ({EPOCHS_SEG} epochs, batch {BATCH_SEG})',
    f'- SAM      : N/A (zero-shot inference)',
    f'- Ensemble : N/A (combines existing models)',
    '',
    '## 3. Strengths',
    '',
    '- YOLOv10 (Agent A): Fast inference, broad multi-class detection (60 xView classes).',
    '- SegFormer (Agent B): Pixel-level masks for region analysis and disaster typing.',
    '- SAM (Agent C): Zero-shot click segmentation — no training needed; useful for interactive annotation UI.',
    '- WBF Ensemble: Reduces false positives by enforcing cross-model agreement.',
    '',
    '## 4. Weaknesses Identified',
    '',
    *[f'- {w}' for w in weaknesses],
    '',
    '## 5. Recommended Improvements',
    '',
    *[f'- {r}' for r in recommendations],
    '',
    '### General hyperparameter playbook for higher mAP / mIoU',
    '',
    '**YOLOv10:**',
    '- `epochs=60, optimizer="AdamW", lr0=0.002, cos_lr=True, warmup_epochs=3`',
    '- `mosaic=1.0, mixup=0.15, copy_paste=0.3, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4`',
    '- Use `yolov8s.pt` or `yolov8m.pt` as base (instead of yolov8n) for +10-15 mAP.',
    '- Set `imgsz=640` if VRAM allows — small objects (Small Car, Pylon) benefit substantially.',
    '- Use multi-scale training: `multi_scale=True`.',
    '',
    '**SegFormer:**',
    '- Switch to `nvidia/segformer-b2-finetuned-ade-512-512` pretrained weights via `transformers`.',
    '- `epochs=50, lr=6e-5, weight_decay=0.01, scheduler="polynomial" (power=1.0)`.',
    '- Use real polygon masks from xView GeoJSON, or SAM-generated pseudo-masks from YOLO boxes.',
    '- Apply Dice + CrossEntropy combined loss with `ignore_index=0` for background.',
    '',
    '**Data:**',
    '- Remove patches with zero annotations (signal-to-noise improvement).',
    '- Class balancing: WeightedRandomSampler over class frequency.',
    '- Increase patch overlap from 64 to 128 px during tiling for boundary coverage.',
    '',
    '## 6. Recommended Best Model',
    '',
    f'**{best_model}** — best detection score (mAP@0.5 = {y.get("mAP50", "N/A")}).',
    '',
    'For the production SkyLogic MAS pipeline, use:',
    '1. **YOLOv10** as primary detector for all 60 classes.',
    '2. **SegFormer** for disaster region segmentation (flood, debris, damage areas).',
    '3. **SAM** in the UI for interactive single-object click annotation.',
    '4. **WBF Ensemble** when both A and B are available — improves robustness.',
    '',
    '## 7. Artefacts',
    '',
    f'- CSV     : `{RESULTS_DIR}/model_comparison_summary.csv`',
    f'- Metrics : `{LOGS_DIR}/*.json`',
    f'- Errors  : `{LOGS_DIR}/*_error.txt` (if any)',
    f'- Plots   : `{PLOTS_DIR}/*.png`',
    f'- Models  : `{RESULTS_DIR}/models/`',
]
report_md = '\n'.join(lines)
report_path = f'{REPORTS_DIR}/training_report.md'
with open(report_path, 'w', encoding='utf-8') as f: f.write(report_md)
print(f'Report saved: {report_path}')
print('\n' + '=' * 70 + '\n')
print(report_md)

In [ ]:
# ── CELL 20: Final Save + Verification ───────────────────────────────────────
import json, shutil
from datetime import datetime
from pathlib import Path

ts = datetime.now().strftime('%Y%m%d_%H%M')

# JSON dump of everything
import numpy as np
def _ser(o):
    if isinstance(o, (np.int64, np.int32)): return int(o)
    if isinstance(o, (np.float64, np.float32)): return float(o)
    raise TypeError(str(type(o)))
with open(f'{RESULTS_DIR}/all_results_{ts}.json', 'w') as f:
    json.dump(MODEL_RESULTS, f, indent=2, default=_ser)

print('=' * 70)
print('FILES SAVED TO DRIVE')
print('=' * 70)
for p in sorted(Path(RESULTS_DIR).rglob('*')):
    if p.is_file():
        kb = p.stat().st_size / 1024
        print(f'  {str(p).replace(RESULTS_DIR, ""):<55} {kb:>8.1f} KB')

print('\nDONE. Run complete. See report at:')
print(f'  {REPORTS_DIR}/training_report.md')